# Module 8: Full Deployment -- The Complete Picture

**Congratulations!** You have built **Aria**, a production AI assistant powered by all 9 Amazon Bedrock AgentCore services. This module reviews the full architecture, verifies every service is active, runs integration tests, and provides cleanup instructions.

## Architecture review

Here is the complete architecture you have built across Modules 2-8:

![Overview](../shared/img/08.drawio.png)

### The 9 AgentCore services + frontend

| # | Service | Module | Role |
|---|---|---|---|
| 1 | **Runtime** | 02 | Hosts the Strands agent, manages invocations and lifecycle |
| 2 | **Code Interpreter** | 03 | Sandboxed Python execution for calculations and data analysis |
| 3 | **Browser Tool** | 03 | Web browsing for current information retrieval |
| 4 | **Memory** | 04 | Short-term (session events) and long-term (preferences, facts, summaries) memory |
| 5 | **Gateway** | 05 | MCP protocol bridge connecting the agent to external APIs |
| 6 | **Identity** | 05 | CUSTOM_JWT authentication forwarding user identity through the stack |
| 7 | **Policy** | 06 | Cedar policy enforcement — deterministic guardrails at the Gateway |
| 8 | **Observability** | 07 | OpenTelemetry tracing to CloudWatch (ADOT SDK + explicit enablement) |
| 9 | **Evaluations** | 07 | LLM-as-judge quality monitoring with custom rubrics |
| -- | **Frontend** | 08 | Web app with streaming chat, sessions, and Cognito auth |

## Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import boto3, json, uuid
from shared import utils

region = utils.get_region()
control = boto3.client("bedrock-agentcore-control", region_name=region)
data_client = boto3.client("bedrock-agentcore", region_name=region)

print(f"Region:  {region}")
print(f"Account: {utils.get_account_id()}")

## Verify all services

Let's confirm that every AgentCore resource is active and properly configured using raw boto3 calls.

In [ ]:
# Verify Runtime
print("=" * 50)
print("  Runtime")
print("=" * 50)

runtime_config = utils.load_config("runtime")
if runtime_config:
    runtime_id = runtime_config["runtime_id"]
    try:
        rt = control.get_agent_runtime(agentRuntimeId=runtime_id)
        print(f"  Name:   {rt.get('agentRuntimeName', 'N/A')}")
        print(f"  ID:     {runtime_id}")
        print(f"  Status: {rt.get('status', 'UNKNOWN')}")
        print(f"  ARN:    {rt.get('agentRuntimeArn', 'N/A')[:80]}...")
    except Exception as e:
        print(f"  Error: {e}")
else:
    print("  Not configured")
print()

In [ ]:
# Verify Memory
print("=" * 50)
print("  Memory")
print("=" * 50)

memory_config = utils.load_config("memory")
if memory_config:
    memory_id = memory_config["memory_id"]
    try:
        mem = control.get_memory(memoryId=memory_id)
        mem_data = mem.get("memory", mem)
        print(f"  Name:       {mem_data.get('name', 'N/A')}")
        print(f"  ID:         {memory_id}")
        print(f"  Status:     {mem_data.get('status', 'UNKNOWN')}")
        strategies = mem_data.get('memoryStrategies', [])
        print(f"  Strategies: {len(strategies)}")
        for s in strategies:
            for key, val in s.items():
                if isinstance(val, dict):
                    print(f"    - {val.get('name', key)}")
    except Exception as e:
        print(f"  Error: {e}")
else:
    print("  Not configured")
print()

In [ ]:
# Verify Gateway
print("=" * 50)
print("  Gateway")
print("=" * 50)

gateway_config = utils.load_config("gateway")
if gateway_config:
    gateway_id = gateway_config["gateway_id"]
    try:
        gw = control.get_gateway(gatewayIdentifier=gateway_id)
        print(f"  Name:     {gw.get('name', 'N/A')}")
        print(f"  ID:       {gateway_id}")
        print(f"  Status:   {gw.get('status', 'UNKNOWN')}")
        print(f"  Protocol: {gw.get('protocolType', 'N/A')}")
        print(f"  Auth:     {gw.get('authorizerType', 'N/A')}")
        policy_cfg = gw.get('policyEngineConfiguration', {})
        if policy_cfg:
            print(f"  Policy:   {policy_cfg.get('mode', 'N/A')} mode")
    except Exception as e:
        print(f"  Error: {e}")
else:
    print("  Not configured")
print()

In [ ]:
# Verify Policy Engine
print("=" * 50)
print("  Policy Engine")
print("=" * 50)

policy_config = utils.load_config("policy")
if policy_config:
    engine_id = policy_config["policy_engine_id"]
    try:
        engine = control.get_policy_engine(policyEngineId=engine_id)
        print(f"  ID:     {engine_id}")
        print(f"  Status: {engine.get('status', 'UNKNOWN')}")
        print(f"  Mode:   {policy_config.get('enforcement_mode', 'N/A')}")

        # List policies
        try:
            policies = control.list_policies(policyEngineId=engine_id)
            policy_list = policies.get("policies", [])
            for p in policy_list:
                print(f"    - {p.get('name', 'unnamed')} ({p.get('status', '?')})")
            print(f"  Total:  {len(policy_list)} Cedar policies")
        except Exception:
            pass
    except Exception as e:
        print(f"  Error: {e}")
else:
    print("  Not configured")
print()

In [ ]:
# Verify Evaluations
print("=" * 50)
print("  Evaluations")
print("=" * 50)

evals_config = utils.load_config("evaluations")
if evals_config:
    custom_evals = evals_config.get("custom_evaluators", {})
    builtin_evals = evals_config.get("builtin_evaluators", [])
    print(f"  Custom evaluators:  {len(custom_evals)}")
    for name, eid in custom_evals.items():
        print(f"    - {name}: {eid}")
    print(f"  Built-in evaluators: {len(builtin_evals)}")
    for name in builtin_evals:
        print(f"    - {name}")
else:
    print("  Not configured")
print()

## Deploy the frontend web application

Now let's deploy a web application so you can interact with Aria through a browser.

The frontend infrastructure (API Gateway, Lambda functions, S3 bucket, CloudFront distribution, DynamoDB sessions table) was **pre-provisioned** by the workshop CloudFormation template. The deploy script wires it all up with the AgentCore resources you created in Modules 2-7:

1. **Configures the Runtime for OAuth** — enables direct HTTPS calls with a Cognito JWT
2. **Adds the POST /chat endpoint** to the pre-provisioned API Gateway, pointing to the Runtime
3. **Updates the history Lambda** with the Memory ID for conversation history retrieval
4. **Uploads frontend files** to S3 with a generated `config.js` containing all endpoints
5. **Invalidates the CloudFront cache** so the new files are served immediately

In [ ]:
# Deploy the frontend (Runtime OAuth + API Gateway + S3/CloudFront + Lambda)
# This takes 3-5 minutes on first deploy.

import sys; sys.path.insert(0, 'scripts')
from deploy_frontend import deploy

frontend_config = deploy()

## Access the web application

The frontend is now live. Open the CloudFront URL below in your browser and log in with the workshop credentials:

- **Username:** `workshop@example.com`
- **Password:** `WorkshopPass123!`

Try these conversations to exercise all of Aria's capabilities:
1. **"Remember that my favorite programming language is Python"** -- tests Memory
2. **"Calculate the first 20 Fibonacci numbers"** -- tests Code Interpreter
3. **"Create a task: Review the AgentCore documentation"** -- tests Gateway + Policy
4. **"What's the latest news about AWS re:Invent?"** -- tests Browser Tool
5. **Start a new session and ask "What's my favorite programming language?"** -- tests cross-session Memory

In [ ]:
# Print the frontend URL and login credentials
frontend_config = utils.load_config("frontend")

if frontend_config:
    url = frontend_config.get("cloudfront_url", "")
    print("=" * 60)
    print("  Aria Web Application")
    print("=" * 60)
    print()
    print(f"  URL:      {url}")
    print()
    print(f"  Username: workshop@example.com")
    print(f"  Password: WorkshopPass123!")
    print()
    print(f"  Open the URL above in your browser to start chatting with Aria.")
    print()
else:
    print("Frontend not deployed yet. Run the deployment cell above first.")

## What you have learned

Across this workshop, you built a production AI assistant from scratch using Amazon Bedrock AgentCore:

| Module | What you did | Key takeaway |
|---|---|---|
| **00** | Set up prerequisites | CloudFormation provisions IAM roles, S3 bucket, Cognito, Task API |
| **01** | Explored the CLI | `agentcore` CLI for local development and testing |
| **02** | Created Runtime | Deployed a Strands agent to AgentCore Runtime |
| **03** | Added Tools | Code Interpreter and Browser Tool -- zero infrastructure |
| **04** | Added Memory | Session summaries, user preferences, and semantic facts |
| **05** | Added Gateway + Identity | MCP bridge to REST APIs with JWT authentication |
| **06** | Added Cedar Policies | Deterministic guardrails that enforce business rules |
| **07** | Added Observability + Evaluations | Auto-tracing and LLM-as-judge quality monitoring |
| **08** | Deployed the web app | Full frontend with streaming chat, sessions, and Cognito auth |

The key architectural insight: each AgentCore service handles one concern, and they compose together cleanly. The agent code itself (Aria) stays simple -- it is a Strands agent with a system prompt and tools. All the production infrastructure (auth, policies, tracing, evaluation) lives in the AgentCore platform layer.

The frontend connects to AgentCore Runtime through an API Gateway streaming proxy. The user's Cognito JWT flows through the entire stack: API Gateway validates it, the Runtime receives it, and the Gateway forwards it to downstream tools. Cedar policies enforce business rules at the Gateway layer without any changes to the agent code.

## Cleanup

When you are done with the workshop, run the cleanup cell below to delete all AgentCore resources (Runtime, Memory, Gateway, Policy Engine).

> **Workshop Studio note:** If you are running this in an AWS Workshop Studio environment, the CloudFormation stack (IAM roles, Cognito, Task API, frontend infrastructure) will be **cleaned up automatically** when your workshop session expires. You only need to delete the AgentCore resources created during the workshop.

For self-hosted environments, you will also need to delete the CloudFormation stacks (`cfn-template` and `code-editor`) manually from the CloudFormation console.

In [ ]:
# Delete AgentCore resources (Runtime, Memory, Gateway, Policy Engine)
import sys; sys.path.insert(0, '..'); sys.path.insert(0, 'scripts')
from cleanup import cleanup

# Uncomment the next line to delete all AgentCore workshop resources:
# cleanup(auto_confirm=True)

## Next steps

Now that you have built a complete AgentCore application, here are some directions to explore:

- **Add more Gateway targets**: Connect additional APIs (calendar, email, CRM) as MCP targets
- **Refine Cedar policies**: Add per-user or per-role policies using JWT claims as Cedar principals
- **Custom evaluators**: Create domain-specific evaluators for your use case (compliance, tone, safety)
- **Multi-agent architectures**: Use AgentCore Gateway to connect multiple agents together
- **Production deployment**: Set up CI/CD pipelines for agent code updates using `deploy_agent.deploy()`

### Resources

- [Amazon Bedrock AgentCore Documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/what-is-bedrock-agentcore.html)
- [Strands Agents SDK](https://github.com/strands-agents/sdk-python)
- [Cedar Language Reference](https://docs.cedarpolicy.com/)
- [AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agents-tools-runtime.html)
- [AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html)
- [AgentCore Memory](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory.html)
- [AgentCore Policy](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/policy.html)
- [AgentCore Observability](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html)
- [AgentCore Evaluations](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/evaluations.html)

In [ ]:
import sys; sys.path.insert(0, '..')
from shared.progress import show

show("08")

---

**Congratulations!** You have completed the Amazon Bedrock AgentCore workshop. Aria is a fully production-ready AI assistant with a web frontend, powered by all 9 AgentCore services, authenticated with Cognito, and protected by Cedar policies.